# 04 · Urgency triage with IndicBERT v2 (owner: Soham Ghodake)

Fine-tunes IndicBERT v2 (2023) on the query set to predict urgency (emergency, clinic, home).
Safety first: class weights favour emergency recall, and the report leads with the **missed-emergency rate**.
In the app, the rules floor (safety_rules.py) still applies: the model can raise urgency, never lower it.
Hinglish is evaluated separately; MuRIL (2021) is the Hinglish comparison.

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn transformers-interpret

In [ ]:
# Clone the repo (or upload it) and move into it.
!git clone https://github.com/<owner>/firstaid-assistant.git 2>/dev/null || true
%cd firstaid-assistant/backend
!python scripts/generate_queries.py

In [ ]:
import json
def load(p): return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
train, dev = load("data/queries/train.jsonl"), load("data/queries/dev.jsonl")
try:
    test = load("data/queries/test.jsonl")
except FileNotFoundError:
    test = []
print(len(train), len(dev), len(test))

In [ ]:
LABELS = ["emergency", "clinic", "home"]
tr = [r for r in train if r.get("urgency")]; dv = [r for r in dev if r.get("urgency")]
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch, numpy as np
name = "ai4bharat/IndicBERTv2-MLM-only"   # MuRIL comparison: "google/muril-base-cased"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=3, id2label=dict(enumerate(LABELS)), label2id={l: i for i, l in enumerate(LABELS)})
mk = lambda rows: Dataset.from_dict({"text": [r["text"] for r in rows], "label": [LABELS.index(r["urgency"]) for r in rows]}).map(lambda b: tok(b["text"], truncation=True, max_length=128), batched=True)
dtr, ddv = mk(tr), mk(dv)
weights = torch.tensor([3.0, 1.0, 1.0])  # missing an emergency costs most
class WTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels"); out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(out.logits, labels, weight=weights.to(out.logits.device))
        return (loss, out) if return_outputs else loss
args = TrainingArguments("out_triage", per_device_train_batch_size=32, learning_rate=3e-5, num_train_epochs=4, fp16=True, save_strategy="no", report_to=[])
WTrainer(model=model, args=args, train_dataset=dtr, tokenizer=tok).train()

In [ ]:
from sklearn.metrics import f1_score
pred = Trainer(model=model, tokenizer=tok).predict(ddv).predictions.argmax(-1)
for lang in ["en", "hi", "hinglish", "mr"]:
    idx = [i for i, r in enumerate(dv) if r["lang"] == lang]
    g = [LABELS.index(dv[i]["urgency"]) for i in idx]; p = [pred[i] for i in idx]
    em = [pp for gg, pp in zip(g, p) if gg == 0]
    missed = sum(pp != 0 for pp in em) / max(1, len(em))
    print(f"{lang:9s} missed_emergency_rate={missed:.3f} macroF1={f1_score(g, p, average='macro'):.3f}")
model.save_pretrained("../training/out/indicbert_triage"); tok.save_pretrained("../training/out/indicbert_triage")

In [ ]:
# F4 check: token attributions for a few emergencies (Section 8.6)
from transformers_interpret import SequenceClassificationExplainer
ex = SequenceClassificationExplainer(model, tok)
for t in ["papa gir gaye, saans nahi le rahe", "सीने में तेज दर्द", "बाबा बेशुद्ध आहेत"]:
    print(t, sorted(ex(t, class_name="emergency"), key=lambda x: -x[1])[:3])